# Health Symptom Diagnosis — Model Training & Evaluation

**Project:** Development and Evaluation of a Machine Learning Based Health Symptom Diagnosis Chatbot Using Decision Tree Algorithm  
**Algorithm:** `sklearn.tree.DecisionTreeClassifier` (criterion=gini)  
**Dataset:** 132-symptom, 41-class medical diagnosis dataset  

## 1. Setup

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'app' / 'artifacts' / 'v1'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Artifacts dir:', ARTIFACTS_DIR)

## 2. Dataset Description

In [ ]:
df_train = pd.read_csv(DATA_DIR / 'Training.csv')
df_test  = pd.read_csv(DATA_DIR / 'Testing.csv')

# Drop unnamed trailing columns
df_train = df_train.loc[:, ~df_train.columns.str.startswith('Unnamed')]
df_test  = df_test.loc[:,  ~df_test.columns.str.startswith('Unnamed')]

print('Training shape:', df_train.shape)
print('Test shape:    ', df_test.shape)
df_train.head(3)

In [ ]:
# Class distribution
class_counts = df_train['prognosis'].value_counts()
print(f'Number of classes: {class_counts.shape[0]}')
print(f'Samples per class — min: {class_counts.min()}, max: {class_counts.max()}, mean: {class_counts.mean():.1f}')

fig, ax = plt.subplots(figsize=(14, 5))
class_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Class Distribution (Training Set)')
ax.set_xlabel('Disease')
ax.set_ylabel('Sample Count')
ax.tick_params(axis='x', rotation=90, labelsize=7)
plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
symptom_list = [c for c in df_train.columns if c != 'prognosis']
print(f'Symptom features: {len(symptom_list)}')

X_train = df_train[symptom_list].values.astype(np.float64)
X_test  = df_test[symptom_list].values.astype(np.float64)

le = LabelEncoder()
y_train = le.fit_transform(df_train['prognosis'])
y_test  = le.transform(df_test['prognosis'])
class_names = list(le.classes_)

# Verify all test labels are known
assert set(df_test['prognosis'].unique()).issubset(set(le.classes_)), \
    'Test set contains unseen labels!'

# Check for missing values
print('Missing values in training set:', df_train.isnull().sum().sum())
print('Feature value range: [{}, {}]'.format(X_train.min(), X_train.max()))

## 4. Decision Tree Training

**Hyperparameter rationale:**
- `criterion='gini'`: Gini impurity is computationally efficient and performs equivalently to entropy on balanced datasets.
- `max_depth=None`: Full-depth trees are appropriate here because the feature space is purely binary (0/1) and each path from root to leaf directly encodes a disease-specific symptom combination. Pruning is not needed when the dataset is clean and features are semantically meaningful.
- `random_state=42`: Reproducibility.

In [ ]:
clf = DecisionTreeClassifier(criterion='gini', max_depth=None, random_state=42)
clf.fit(X_train, y_train)
print('Tree depth:', clf.get_depth())
print('Tree leaves:', clf.get_n_leaves())

## 5. 5-Fold Stratified Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring='accuracy')

for i, s in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {s:.4f}')
print(f'  Mean: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(1, 6), cv_scores, color='steelblue')
ax.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Mean {cv_scores.mean():.4f}')
ax.set_xticks(range(1, 6))
ax.set_xlabel('Fold')
ax.set_ylabel('Accuracy')
ax.set_title('5-Fold CV Accuracy')
ax.legend()
ax.set_ylim(0.9, 1.01)
plt.tight_layout()
plt.show()

## 6. Confusion Matrix

In [ ]:
y_pred = clf.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(18, 15))
sns.heatmap(
    cm,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax,
    linewidths=0.3,
)
ax.set_title('Confusion Matrix — Test Set', fontsize=14)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
ax.tick_params(axis='x', rotation=90, labelsize=7)
ax.tick_params(axis='y', rotation=0, labelsize=7)
plt.tight_layout()
plt.show()

## 7. Classification Report

In [ ]:
test_accuracy = (y_pred == y_test).mean()
print(f'Test accuracy: {test_accuracy:.4f}\n')
print(classification_report(y_test, y_pred, target_names=class_names))

## 8. Feature Importance — Top 20 Symptoms

In [ ]:
importances = clf.feature_importances_
top_n = 20
top_idx = np.argsort(importances)[::-1][:top_n]
top_names = [symptom_list[i].replace('_', ' ') for i in top_idx]
top_vals  = importances[top_idx]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_names[::-1], top_vals[::-1], color='steelblue')
ax.set_xlabel('Gini Importance')
ax.set_title(f'Top {top_n} Symptom Importances')
plt.tight_layout()
plt.show()

## 9. Decision Tree Visualisation (max_depth=3)

In [ ]:
fig, ax = plt.subplots(figsize=(24, 8))
plot_tree(
    clf,
    max_depth=3,
    feature_names=[s.replace('_', ' ') for s in symptom_list],
    class_names=class_names,
    filled=True,
    rounded=True,
    fontsize=7,
    ax=ax,
)
ax.set_title('Decision Tree (first 3 levels)', fontsize=14)
plt.tight_layout()
plt.show()

## 10. Comparison with Random Forest

A Random Forest is included as a baseline to validate the Decision Tree choice.

**Justification for choosing DT over RF:**
- The dataset has perfect separability (all class-symptom combinations are deterministic), so a single DT already achieves near-perfect accuracy without ensembling.
- A DT is fully interpretable: the decision path for any prediction can be read directly, which is important for academic transparency and for building user trust in a medical context.
- RF adds computational overhead and loses interpretability without providing a meaningful accuracy gain on this dataset.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_cv = cross_val_score(rf, X_train, y_train, cv=cv, scoring='accuracy')
rf_test_acc = (rf.predict(X_test) == y_test).mean()

dt_test_acc = test_accuracy
dt_cv_mean  = cv_scores.mean()

print('Model            | Test Acc | CV Mean')
print('-' * 42)
print(f'Decision Tree    | {dt_test_acc:.4f}   | {dt_cv_mean:.4f}')
print(f'Random Forest    | {rf_test_acc:.4f}   | {rf_cv.mean():.4f}')

fig, ax = plt.subplots(figsize=(6, 3))
models = ['Decision Tree', 'Random Forest']
test_accs = [dt_test_acc, rf_test_acc]
cv_means  = [dt_cv_mean, rf_cv.mean()]
x = np.arange(len(models))
w = 0.35
ax.bar(x - w/2, test_accs, w, label='Test Accuracy')
ax.bar(x + w/2, cv_means,  w, label='CV Mean Accuracy')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0.85, 1.01)
ax.set_ylabel('Accuracy')
ax.set_title('Decision Tree vs Random Forest')
ax.legend()
plt.tight_layout()
plt.show()

## 11. Export Artifacts

In [ ]:
joblib.dump(clf, ARTIFACTS_DIR / 'model.joblib')
joblib.dump(le,  ARTIFACTS_DIR / 'label_encoder.joblib')

metadata = {
    'symptom_list':    symptom_list,
    'class_names':     class_names,
    'model_version':   '1.0.0',
    'accuracy':        round(float(test_accuracy), 6),
    'cv_mean_accuracy': round(float(cv_scores.mean()), 6),
}
with open(ARTIFACTS_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# disease_info.json — load from optional CSVs if present
disease_info = {d: {'description': '', 'precautions': []} for d in class_names}

desc_path = DATA_DIR / 'symptom_Description.csv'
prec_path = DATA_DIR / 'symptom_precaution.csv'

if desc_path.exists():
    df_desc = pd.read_csv(desc_path)
    for _, row in df_desc.iterrows():
        d = str(row.iloc[0]).strip()
        if d in disease_info:
            disease_info[d]['description'] = str(row.iloc[1]).strip()
    print('Loaded descriptions')

if prec_path.exists():
    df_prec = pd.read_csv(prec_path)
    for _, row in df_prec.iterrows():
        d = str(row.iloc[0]).strip()
        if d in disease_info:
            disease_info[d]['precautions'] = [
                str(row.iloc[i]).strip()
                for i in range(1, len(row))
                if str(row.iloc[i]).strip() not in ('', 'nan')
            ]
    print('Loaded precautions')

with open(ARTIFACTS_DIR / 'disease_info.json', 'w') as f:
    json.dump(disease_info, f, indent=2)

print('Artifacts written to', ARTIFACTS_DIR)
print('  model.joblib')
print('  label_encoder.joblib')
print('  metadata.json')
print('  disease_info.json')